# AlphaEarth Foundations (AEF) Annual Satellite Embeddings

**Living Earth - Cube in a Box Demo Series**

---

*   **Objective:** Load and visualize the AlphaEarth Foundations (AEF) annual 64-band satellite embeddings at 10 m, including a PCA-based RGB preview.
*   **Products used:** [`aef_annual`](http://localhost/explorer/products/aef_annual)
*   **Source:** [Source Cooperative — tge-labs/aef](https://source.coop/tge-labs/aef) · [Google Earth Engine catalog](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL)

---

## Background

The AlphaEarth Foundations (AEF) annual Satellite Embedding dataset (produced by Google and Google DeepMind, mirrored on Source Cooperative by Taylor Geospatial Engine Labs) provides global, analysis-ready **64-dimensional geospatial embeddings** at **10 m** resolution, one vector per pixel per year (2017–2025). Each embedding assimilates optical and thermal imagery (Sentinel-2, Landsat), radar, LiDAR, elevation, climate and other sources into a unit-length vector that summarizes annual surface conditions.

The stored values are `int8` (`A00`–`A63`, nodata `-128`): a quantized form of the unit-length embedding. The de-quantization mapping that restores the float unit-length vectors is documented at the Source Cooperative page above; this notebook works directly on the quantized bands for a quick visual preview.

## Description

This notebook demonstrates how to load the `aef_annual` product for a small area of interest, inspect its 64 measurements, preview a few individual embedding bands, and render a PCA-based RGB composite that reduces the 64-D vectors to three channels for visualization.

***

In [ ]:
import sys
sys.path.insert(1, './utils/')

In [ ]:
# reload module before executing code
%load_ext autoreload
%autoreload 2

import time

import datacube
import numpy as np
import matplotlib.pyplot as plt

from utils.le_aef import ensure_aef_gdal_env
from utils.le_dc import get_product_bbox
from utils.le_mapping import bbox_to_polygon, display_crosshair, get_utm_epsg_code, MapHandler
from utils.le_tools import style_output_cells

ensure_aef_gdal_env()

### Connect to the datacube

In [ ]:
dc = datacube.Datacube(app='AEF_annual')

## Describe product measurements

In [ ]:
dc.list_measurements().loc['aef_annual']

## Load AEF annual embeddings

AEF is served as **unsigned HTTPS COGs from Source Cooperative** — no credentials and no `patch_url` are needed.

The product has 64 `int8` bands (`A00`–`A63`, nodata `-128`) at 10 m, one layer per year. Before loading, this notebook calls `ensure_aef_gdal_env()` from `utils.le_aef` to tune GDAL for anonymous COG range reads (merged consecutive ranges, vsicurl block cache). We pass `dask_chunks={'x': 2048, 'y': 2048, 'time': 1}` so the 64 bands are fetched in parallel — the main win for medium/large areas of interest. Because the bands are quantized embedding vectors, any reprojection uses **nearest** resampling — never bilinear/cubic, which would invent meaningless intermediate vectors.

> **Note:** the stored `int8` values are a quantization of the unit-length embedding. For analysis (similarity, clustering, classification) de-quantize and renormalize to unit length as described at [source.coop/tge-labs/aef](https://source.coop/tge-labs/aef). This preview works on the raw quantized bands.

In [ ]:
# Check if default bbox is contained within the datacube
# and allow user to draw bbox if not.

product = 'aef_annual'

# configure a default bounding box and visualize it
lat, lon = 22.821, 28.518
buffer = 0.05
default_bbox = (lon - buffer, lat - buffer, lon + buffer, lat + buffer)

product_bbox = get_product_bbox(dc, product, split_size=10, stability_threshold=4)

is_contained = (default_bbox[0] >= product_bbox[0] and
               default_bbox[1] >= product_bbox[1] and
               default_bbox[2] <= product_bbox[2] and
               default_bbox[3] <= product_bbox[3]
              )

In [ ]:
# Create an instance of MapHandler
map_handler = MapHandler()
m, drc = map_handler.create_map(vect=[bbox_to_polygon(default_bbox), bbox_to_polygon(product_bbox)],
                                draw_rect=True)
display(m)

# append crosshair
time.sleep(2)  # make sure m is fully displayed
display_crosshair()

In [ ]:
# Warn in case of full AoI
aoi_poly = map_handler.aoi_tupple

if aoi_poly is None:
    aoi_poly = tuple(default_bbox)
    if not is_contained:
        style_output_cells('salmon', border_color='red', border_width='2px')
        print('The area of interest polygon is located outside of the product extent.' + \
              '\nPlease draw a new area of interest in the previous cell.')
    else:
        # When is_contained is True and no polygon drawn - this is actually OK!
        style_output_cells()
        print('Default area of interest is contained within the product extent, but you can still draw another one in the previous cell.')
else:
    # A polygon was drawn
    style_output_cells()
    print('Custom area of interest polygon has been created.')

In [ ]:
# get EPSG code for the center of the AoI

epsg_code = get_utm_epsg_code((aoi_poly[1] + aoi_poly[3]) / 2, (aoi_poly[0] + aoi_poly[2]) / 2)

In [ ]:
times = [ds.time.begin for ds in dc.find_datasets(product=product)]
first_year = sorted({t.year for t in times})

In [ ]:
# Load the AEF annual embeddings for the area of interest.
# No patch_url: AEF hrefs are unsigned HTTPS COGs

query = {
    'product': product,
    'x': (aoi_poly[0], aoi_poly[2]),
    'y': (aoi_poly[1], aoi_poly[3]),
    'time': str(first_year[0]),
    'output_crs': f"EPSG:{epsg_code}",
    'resolution': 10,
    'resampling': 'nearest',
    'dask_chunks': {'x': 2048, 'y': 2048, 'time': 1},
}

ds = dc.load(**query)
print(ds)

## Preview individual embedding bands

Each band `A00`–`A63` is one dimension of the 64-D embedding. Individual bands carry no direct physical meaning, but previewing a few gives a sense of the spatial structure encoded by the model.

In [ ]:
# Preview a few individual embedding bands for the first (only) time step

da = ds["A00"].isel(time=0)
nodata = da.attrs.get('nodata', -128)

bands = ['A00', 'A01', 'A02']
fig, axes = plt.subplots(1, len(bands), figsize=(5 * len(bands), 5))
for ax, b in zip(axes, bands):
    arr = ds[b].isel(time=0).values.astype('float32')
    arr[arr == nodata] = np.nan
    im = ax.imshow(arr, cmap='viridis')
    ax.set_title(b)
    ax.set_aspect('equal')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## PCA RGB composite

To visualize the 64-D embeddings as a single image, reduce the 64 bands to 3 channels using PCA (via numpy SVD) and display the first three principal components as an RGB composite. This is a common way to inspect embedding structure with the naked eye.

In [ ]:
# PCA -> RGB composite for the first time step.
# Stack the 64 bands into (n_pixels, 64), drop nodata, compute the first 3 PCs.
band_names = [f'A{i:02d}' for i in range(64)]
stack = np.stack([ds[b].isel(time=0).values for b in band_names]).astype('float32')  # (64, H, W)
nodata = ds['A00'].attrs.get('nodata', -128)

H, W = stack.shape[1], stack.shape[2]
valid = stack[0] != nodata  # nodata mask is shared across bands
X = stack[:, valid].T  # (N, 64)

# Centre and compute first 3 principal components via SVD (no sklearn dependency)
Xc = X - X.mean(axis=0, keepdims=True)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pcs = U[:, :3] * S[:3]  # (N, 3) scores

# Per-channel min-max stretch to [0, 1]
rgb = np.zeros((H, W, 3), dtype='float32')
rgb_flat = np.full((H * W, 3), np.nan, dtype='float32')
pcs_scaled = (pcs - pcs.min(axis=0)) / np.ptp(pcs, axis=0)
rgb_flat[valid.ravel()] = pcs_scaled
rgb = rgb_flat.reshape(H, W, 3)

plt.figure(figsize=(8, 8))
plt.imshow(rgb)
plt.title(f'AEF PCA RGB — {str(ds.time.values[0])[:10]}')
plt.axis('off')
plt.show()

***

## Additional information

**Compatible datacube version:**

In [ ]:
print(datacube.__version__)

**Last tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')

In [ ]:
!pip freeze